In [1]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
pypdf

In [2]:
%pip install pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_filename = "Gesture_AI_Project_Workflow.pdf"
loader = PyPDFLoader(pdf_filename)
documents = loader.load()
print(f"Total pages loaded: {len(documents)}")

/tmp/ipykernel_1156/534534059.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages loaded: 5


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
texts = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(texts)}")

Total chunks created: 12


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)
print('document ingested')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

document ingested


In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "HUGGINGFACEHUB_API_TOKEN"

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm_endpoint)

print("LLM ready!")

In [7]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definitely do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=False
)

query = "How many gestures are in the vocabulary and what are the three categories?"
qa.invoke(query)

{'query': 'How many gestures are in the vocabulary and what are the three categories?',
 'result': 'I don\'t know. The text does not explicitly mention the number of gestures in the vocabulary, and the three categories are referred to as "Emergency/Health gestures", "Daily Needs gestures", and "Common Phrases gestures" but it does not mention a specific number of gestures in each category.'}

In [8]:
qa_debug = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

query = "How many gestures are in the vocabulary and what are the three categories?"
result = qa_debug.invoke(query)

print("ANSWER:", result['result'])
print("\n--- RETRIEVED CHUNKS ---")
for i, doc in enumerate(result['source_documents']):
    print(f"\nChunk {i+1}:")
    print(doc.page_content[:300])

ANSWER: I don't know. The text only mentions that there are three categories (Emergency/Health gestures, Daily Needs gestures, and Common Phrases gestures), but it doesn't provide a total number of gestures. Additionally, it only mentions that each person is responsible for collecting and recording 6 gestures, but it doesn't provide a total count for all three categories.

--- RETRIEVED CHUNKS ---

Chunk 1:
here.
Camera (live video)
↓
MediaPipe Hands → extracts 21 hand landmarks (x, y, z) per frame
↓
Sequence of landmarks (30 frames ≈ 1 gesture)
↓
Trained GRU/LSTM model → classiﬁes which gesture it is
↓
Gesture label → looked up in a dictionary → Urdu text
↓
Urdu Text-to-Speech (TTS) engine
↓
Speaker o

Chunk 2:
Person AData collection + labeling (Emergency/Health gestures) + overall dataset coordination
Person BML model — MediaPipe landmark extraction + GRU/LSTM training (Daily Needs gestures)
Person CBackend, Urdu Text-to-Speech integration, frontend/demo UI (Common Phrases gestures)

In [9]:
for i, chunk in enumerate(texts):
    if "18 gestures" in chunk.page_content or "Vocabulary" in chunk.page_content:
        print(f"Found in Chunk {i}:")
        print(chunk.page_content)
        print("---")

Found in Chunk 1:
This project builds a real-time system that:
Watches a person's hand gestures through a webcam
Recognizes which PSL sign is being made
Converts that sign into Urdu text
Speaks the Urdu text out loud
2. Scope Decisions
Isolated gestures only (one sign at a time), not full continuous sentence signing — this keeps the project
realistic for the timeline.
Vocabulary: 18 gestures, split into 3 categories of 6 each (one category per team member):
Emergency/Health: Help, Pain, Doctor, Sick, Ambulance, Medicine
Daily Needs: Water, Food, Bathroom, Home, Sleep, Hungry
Common Phrases: Thank you, Yes, No, Sorry, I'm ﬁne, I don't understand
3. Team Roles
Person Responsibility
Person AData collection + labeling (Emergency/Health gestures) + overall dataset coordination
---
Found in Chunk 6:
School visit (validation) 5–10 —
Total per gesture ~50–70 samples
Overall dataset target: ~800–1000 video clips across all 18 gestures.
Data Augmentation (to stretch a small dataset)
Mirror ﬂip (

In [10]:
retriever = docsearch.as_retriever(search_kwargs={"k": 8})

qa_v2 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=False
)

query = "How many gestures are in the vocabulary and what are the three categories?"
qa_v2.invoke(query)

{'query': 'How many gestures are in the vocabulary and what are the three categories?',
 'result': "According to the text, the vocabulary consists of 18 gestures, split into 3 categories of 6 each. The three categories are:\n\n1. Emergency/Health: Help, Pain, Doctor, Sick, Ambulance, Medicine\n2. Daily Needs: Water, Food, Bathroom, Home, Sleep, Hungry\n3. Common Phrases: Thank you, Yes, No, Sorry, I'm fine, I don't understand"}

In [11]:
%pip install gradio -q

In [12]:
import gradio as gr

def answer_question(query):
    result = qa_v2.invoke(query)
    return result['result']

demo = gr.Interface(
    fn=answer_question,
    inputs="text",
    outputs="text",
    title="Gesture AI Project — Q&A Assistant",
    description="Apne Gesture AI (PSL to Urdu Speech) project workflow document se koi bhi sawal poochein.",
    examples=[
        "How many gestures are in the vocabulary and what are the three categories?",
        "What tech stack is used for hand tracking and the model?",
        "What is the build plan for each day?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55224b5b6162c61cf8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://55224b5b6162c61cf8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
